# Facial Expression Recognition on FER2013: leak-free training & evaluation

**Baseline:** CNN architecture from
[shivamsingh96/Facial-Expression-Detection-using-CNN-Real-Time-Recognition-with-Webcam](https://github.com/shivamsingh96/Facial-Expression-Detection-using-CNN-Real-Time-Recognition-with-Webcam).

**What this notebook changes (my contribution):**
1. **Fixes data leakage.** The original oversampled the *whole* dataset and *then* split it, so duplicated images
   ended up in both train and test. Its reported 81.3% accuracy is measured partly on memorised images.
   Section 3 measures how big that leak is.
2. **Uses the official FER2013 splits:** `Training` → train, `PublicTest` → validation (model selection),
   `PrivateTest` → final test (touched exactly once).
3. **Handles class imbalance without leaking:** *square-root* balanced class weights on the training split only
   (full balanced weights put ~9x on *disgust*, which destabilised training and made the model over-predict it).
4. **Stronger but still simple CNN** (VGG-style, 4 blocks of 2 convs + global average pooling). The baseline
   architecture underfits (≈58% train accuracy). Set `ARCH = "baseline"` to train the original network instead.
5. **Data augmentation** (flip, rotate, shift, zoom), **gradient clipping**, **early stopping**, **LR scheduling**,
   best-checkpoint restore.
6. **Class-prior correction** for inference: undoes the residual class-weight bias. Its strength is chosen on the
   validation set, then applied once to the test set.
7. **Proper reporting:** accuracy, macro-F1, per-class report, normalised confusion matrix on the untouched test set.

Run on Colab with a GPU: *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.

## 1. Setup

In [ ]:
import os, sys, json, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

IN_COLAB = "google.colab" in sys.modules
ARCH = os.environ.get("FER_ARCH", "improved")   # "improved" (default) or "baseline"
QUICK_TEST = os.environ.get("FER_QUICK_TEST") == "1"   # used only to smoke-test this notebook

SEED = 42
keras.utils.set_random_seed(SEED)
EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
IMG = 48
print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU") or "none")

## 2. Load FER2013

Set `CSV_PATH` to wherever `fer2013.csv` lives. Options:
* **Google Drive** (default below): put the CSV in your Drive and adjust the path.
* **Upload**: set `CSV_PATH = None`, a file picker appears (slow for ~290 MB).

The CSV has columns `emotion`, `pixels`, `Usage` (`Training` / `PublicTest` / `PrivateTest`).

In [ ]:
CSV_PATH = "/content/drive/MyDrive/fer2013.csv"   # <-- change if needed

if IN_COLAB and CSV_PATH and CSV_PATH.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if IN_COLAB and CSV_PATH is None:
    from google.colab import files
    up = files.upload()
    CSV_PATH = next(iter(up))
CSV_PATH = os.environ.get("FER_CSV", CSV_PATH)

data = pd.read_csv(CSV_PATH)
print(data.shape)
print(data["Usage"].value_counts())
data.head()

In [ ]:
# pixels string -> uint8 array (N, 48, 48)
X_all = np.stack([np.asarray(p.split(), dtype=np.uint8) for p in data["pixels"]]).reshape(-1, IMG, IMG)
y_all = data["emotion"].to_numpy()
usage = data["Usage"].to_numpy()

counts = pd.Series(y_all).value_counts().sort_index()
plt.figure(figsize=(9, 4))
sns.barplot(x=[EMOTIONS[i] for i in counts.index], y=counts.values, color="steelblue")
plt.title("FER2013 class distribution (all splits)"); plt.ylabel("images"); plt.show()
print(dict(zip(EMOTIONS, counts.values)))

## 3. How much did the original evaluation leak?

Re-create the original protocol (oversample everything to 8,989 per class, then a random 90/10 split)
**on indices only** and count how many "test" rows are exact copies of a training row.
No training is needed to show the problem.

In [ ]:
rng = np.random.default_rng(45)
idx = np.arange(len(y_all))
max_n = np.bincount(y_all).max()
over = np.concatenate([np.concatenate([idx[y_all == c],
                                       rng.choice(idx[y_all == c], max_n - (y_all == c).sum(), replace=True)])
                       for c in range(7)])
perm = rng.permutation(len(over))
n_test = int(round(0.1 * len(over)))
leak_test, leak_train = over[perm[:n_test]], over[perm[n_test:]]
in_train = np.isin(leak_test, leak_train)
leak_by_class = {EMOTIONS[c]: round(100 * in_train[y_all[leak_test] == c].mean(), 1) for c in range(7)}
print(f"Original protocol: {100 * in_train.mean():.1f}% of test rows also appear in the training set")
print("Per class (% of test rows leaked):", leak_by_class)

In [ ]:
# FER2013 also contains some exact duplicate images *across* its official splits.
# Hash every image and remove from training anything that also appears in validation/test.
hashes = np.array([hashlib.md5(x.tobytes()).hexdigest() for x in X_all])
eval_hashes = set(hashes[usage != "Training"])
train_dup = (usage == "Training") & np.isin(hashes, list(eval_hashes))
print(f"Training images that are exact copies of a val/test image: {train_dup.sum()} (removed)")

## 4. Leak-free splits

In [ ]:
tr = (usage == "Training") & ~train_dup
va = usage == "PublicTest"
te = usage == "PrivateTest"

def prep(mask):
    return (X_all[mask].astype("float32") / 255.0)[..., None], y_all[mask]

x_train, y_train = prep(tr)
x_val, y_val = prep(va)
x_test, y_test = prep(te)

if QUICK_TEST:   # tiny subset just to check the notebook runs end-to-end
    x_train, y_train = x_train[:512], y_train[:512]
    x_val, y_val = x_val[:128], y_val[:128]

print("train", x_train.shape, "| val", x_val.shape, "| test", x_test.shape)

# Imbalance: weight the loss instead of duplicating images (computed on TRAIN ONLY).
# Square root of the "balanced" weights: full weights give disgust ~9x, which caused loss spikes
# and heavy over-prediction of disgust; sqrt keeps it at ~3x.
cw = compute_class_weight("balanced", classes=np.arange(7), y=y_train) ** 0.5
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("class weights:", {EMOTIONS[i]: round(w, 2) for i, w in class_weight.items()})

## 5. Augmentation + input pipeline

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),          # about +/- 18 degrees
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name="augment")

BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE
sw_train = np.array([class_weight[c] for c in y_train], dtype="float32")

def make_ds(x, y, sw=None, train=False):
    y1 = keras.utils.to_categorical(y, 7)
    ds = tf.data.Dataset.from_tensor_slices((x, y1) if sw is None else (x, y1, sw))
    if train:
        ds = ds.shuffle(len(x), seed=SEED)
    ds = ds.batch(BATCH)
    if train:
        ds = ds.map(lambda *t: (augment(t[0], training=True),) + t[1:], num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

ds_train = make_ds(x_train, y_train, sw_train, train=True)
ds_val = make_ds(x_val, y_val)

# show a few augmented samples
xb = next(iter(ds_train))[0][:12].numpy()
plt.figure(figsize=(12, 2))
for i in range(12):
    plt.subplot(1, 12, i + 1); plt.imshow(xb[i, ..., 0], cmap="gray"); plt.axis("off")
plt.suptitle("augmented training samples"); plt.show()

## 6. Model

* `baseline`: the original repository's network (5 convs, Dense(250)); about 0.8M parameters. It underfits.
* `improved` (default): VGG-style, 4 blocks × 2 convs (64-128-256-512), BatchNorm, global average pooling,
  Dense(256); about 4.8M parameters. Lighter dropout, because the problem was underfitting, not overfitting.

In [ ]:
def build_model(num_classes=7):
    def conv_block(x, filters, padding, idx):
        sfx = "" if idx == 0 else f"_{idx}"
        x = layers.Conv2D(filters, (3, 3), padding=padding, name=f"conv2d{sfx}")(x)
        x = layers.BatchNormalization(axis=3, name=f"batch_normalization{sfx}")(x)
        return layers.Activation("relu", name=f"activation{sfx}")(x)

    inp = keras.Input((IMG, IMG, 1), name="input_1")
    x = conv_block(inp, 32, "valid", 0)
    x = layers.Dropout(0.25, name="dropout")(x)
    x = conv_block(x, 64, "same", 1)
    x = layers.MaxPooling2D((2, 2), name="max_pooling2d")(x)
    x = conv_block(x, 64, "valid", 2)
    x = layers.Dropout(0.25, name="dropout_1")(x)
    x = conv_block(x, 128, "same", 3)
    x = layers.MaxPooling2D((2, 2), name="max_pooling2d_1")(x)
    x = conv_block(x, 128, "valid", 4)
    x = layers.MaxPooling2D((2, 2), name="max_pooling2d_2")(x)
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(250, activation="relu", name="dense")(x)
    x = layers.Dropout(0.5, name="dropout_2")(x)
    out = layers.Dense(num_classes, activation="softmax", name="dense_1")(x)
    return keras.Model(inp, out, name="fer_cnn")

def build_improved(num_classes=7):
    def block(x, filters, drop):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        return layers.Dropout(drop)(x) if drop else x

    inp = keras.Input((IMG, IMG, 1))
    x = block(inp, 64, 0.0)
    x = block(x, 128, 0.0)
    x = block(x, 256, 0.25)
    x = block(x, 512, 0.25)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inp, out, name="fer_vgg")

model = build_improved() if ARCH == "improved" else build_model()
print("ARCH =", ARCH)
model.summary()

## 7. Train (validation = PublicTest; the test set is not used here)

In [ ]:
EPOCHS = int(os.environ.get("FER_EPOCHS", 2 if QUICK_TEST else 100))
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),   # clipping prevents the loss spikes
              loss="categorical_crossentropy", metrics=["accuracy"])
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint("fer_model.keras", monitor="val_accuracy", save_best_only=True),
]
t0 = time.time()
history = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks, verbose=2)
print(f"training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="validation (PublicTest)")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="validation (PublicTest)")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 8. Final evaluation on PrivateTest (run once)

Note: training accuracy is *lower* than validation accuracy early on because augmentation and dropout are
active during training only. That is expected and is not a sign of leakage.

In [ ]:
best = keras.models.load_model("fer_model.keras")

# Class-prior correction: multiply probabilities by w_c^-a. Choose a on VALIDATION (PublicTest) only.
val_probs = best.predict(x_val if not QUICK_TEST else prep(va)[0], batch_size=256, verbose=0)
y_val_full = y_val if not QUICK_TEST else prep(va)[1]
grid = {}
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    grid[a] = f1_score(y_val_full, (val_probs * cw ** -a).argmax(1), average="macro")
best_a = max(grid, key=grid.get)
corr = cw ** -best_a
corr = corr / corr.max()
print("validation macro-F1 by correction strength:", {k: round(v, 3) for k, v in grid.items()}, "-> a =", best_a)

raw_probs = best.predict(x_test, batch_size=256, verbose=0)
raw_acc = accuracy_score(y_test, raw_probs.argmax(1))
probs = raw_probs * corr
y_pred = probs.argmax(1)
print(f"PrivateTest accuracy without correction: {raw_acc * 100:.2f}%")

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"PrivateTest accuracy: {acc * 100:.2f}%   macro-F1: {macro_f1:.3f}   (n={len(y_test)})")
print(classification_report(y_test, y_pred, target_names=EMOTIONS, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Normalised confusion matrix (PrivateTest)")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Side-by-side comparison for the report / slides
report = classification_report(y_test, y_pred, target_names=EMOTIONS, output_dict=True, zero_division=0)
metrics = {
    "protocol": "official splits: Training / PublicTest (val) / PrivateTest (test)",
    "arch": ARCH,
    "params": int(best.count_params()),
    "test_accuracy_without_correction": round(raw_acc, 4),
    "inference_prior_correction": [round(float(v), 5) for v in corr],
    "inference_prior_correction_power": best_a,
    "test_accuracy": round(acc, 4),
    "test_macro_f1": round(macro_f1, 4),
    "per_class_f1": {e: round(report[e]["f1-score"], 3) for e in EMOTIONS},
    "epochs_trained": len(h["loss"]),
    "original_leaky_accuracy_reported": 0.8131,
    "original_protocol_leak_pct_of_test_rows": round(100 * in_train.mean(), 1),
    "train_val_test_sizes": [int(len(y_train)), int(len(y_val)), int(len(y_test))],
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

comp = pd.DataFrame({
    "Original (leaky)": [0.8131, "random split after oversampling", f"{100 * in_train.mean():.0f}% of test rows seen in training"],
    "This notebook": [round(acc, 4), "official FER2013 PrivateTest", "0% (plus cross-split duplicates removed)"],
}, index=["test accuracy", "test set", "train/test overlap"])
comp

## 9. Export for the webcam demo

Download **`fer_model.keras`** and put it in `models/v1/` (or a new `models/v2/`). The demo picks it up automatically
(it falls back to the old `fer.h5` if the new file is missing).

In [ ]:
best.save_weights("fer.weights.h5")          # weights-only backup, loadable by fer_model.build_model()
if IN_COLAB:
    from google.colab import files
    for f in ["fer_model.keras", "fer.weights.h5", "metrics.json", "training_curves.png", "confusion_matrix.png"]:
        files.download(f)